# Estimando um Sistema de Demanda Residencial de Energia com Regressão Aparentemente Não Relacionada

## Resumo executivo

Uma concessionária regional estima um **sistema de demanda de energia** residencial de duas equações — participações orçamentárias de eletricidade e gás natural como funções do preço próprio, do preço cruzado e da renda domiciliar — usando **PROC SYSLIN** com o método de regressão aparentemente não relacionada (SUR). As duas equações de participação compartilham um orçamento domiciliar comum, portanto seus erros são correlacionados de forma cruzada; o SUR estima as equações conjuntamente para explorar essa correlação, recuperando efeitos de preço próprio e cruzado coerentes em sinal e fornecendo a covariância entre equações e os testes de restrição de que um analista de demanda necessita.

## Fontes de dados

| Conjunto de dados | Linhas | Granularidade | Variáveis-chave | Descrição |
|---------|------|-------|---------------|-------------|
| `energy` | 60 | uma linha por observação mensal de mercado | `month`, `lp_elec`, `lp_gas`, `lincome`, `w_elec`, `w_gas` | Painel sintético mensal do mercado residencial de energia. `lp_elec`/`lp_gas` são logaritmos dos preços reais de eletricidade e gás natural; `lincome` é o logaritmo da renda domiciliar real; `w_elec`/`w_gas` são as participações orçamentárias de despesa para eletricidade e gás natural, geradas a partir de uma estrutura de demanda estilo AIDS conhecida mais ruído correlacionado entre equações. |

# Estimando um Sistema de Demanda Residencial de Energia com Regressão Aparentemente Não Relacionada

Uma concessionária regional de gás e eletricidade quer entender como as famílias realocam seu orçamento de energia entre **eletricidade** e **gás natural** conforme os preços relativos e a renda mudam. O arcabouço natural é um **sistema de demanda**: um conjunto de equações de participação orçamentária estimadas conjuntamente.

Duas características fazem da estimação conjunta a ferramenta certa:

- As equações de participação recorrem a um orçamento domiciliar comum, portanto seus erros são **correlacionados de forma cruzada** — quando uma família gasta mais em eletricidade, gasta menos em gás. Estimar as equações juntas com **regressão aparentemente não relacionada (SUR)** usa essa correlação para ganhar eficiência.
- A teoria econômica impõe **restrições lineares** (adição, homogeneidade, simetria) que um estimador de sistema pode fazer valer diretamente.

O `PROC SYSLIN` com a opção `SUR` foi construído exatamente para esta situação. Ele ajusta cada equação de participação, estima a covariância dos erros entre equações e reajusta o sistema com essa covariância para entregar elasticidades eficientes e coerentes com a teoria — junto com a matriz de covariância entre modelos e os testes conjuntos de restrição.

Neste notebook nós:

1. Geramos um painel sintético mensal do mercado de energia a partir de uma estrutura de participação conhecida.
2. Ajustamos um sistema de participação de duas equações com SUR.
3. Comparamos o ajuste conjunto SUR com MQO equação a equação.
4. Impomos uma restrição de homogeneidade e lemos seu teste F conjunto.
5. Extraímos a tabela de coeficientes para o relato de elasticidades.

## Passo 1 — Gerar um painel sintético do mercado de energia

Simulamos 60 observações mensais de um pequeno **sistema de demanda de energia de dois bens** (eletricidade e gás natural). O processo gerador dos dados é um modelo de participação orçamentária linearizado, estilo AIDS:

```
w_elec = a1 + g11*lp_elec + g12*lp_gas + b1*lincome + e1
w_gas  = a2 + g21*lp_elec + g22*lp_gas + b2*lincome + e2
```

Construímos deliberadamente **correlação entre equações**: quando as famílias gastam mais em eletricidade, gastam menos em gás, portanto `e1` e `e2` são negativamente correlacionados. Um fator de preço comum do mercado de energia também move ambos os logaritmos de preço juntos. Essas são as características que recompensam a estimação conjunta SUR em vez de ajustar cada equação isoladamente.

In [1]:
DADOS energy;
   CHAMAR streaminit(70123);

   /* True structural coefficients (linearized AIDS share system) */
   a1  = 0.55;  g11 =  0.12; g12 = -0.08; b1 = -0.030;
   a2  = 0.45;  g21 = -0.08; g22 =  0.10; b2 = -0.025;

   FAZER month = 1 ATÉ 60;
      /* A common energy-market price factor drives BOTH prices,
         creating the collinearity that makes the problem ill-posed. */
      energy_factor = rand('NORMAL', 0, 0.15);

      lp_elec = 4.6 + energy_factor + rand('NORMAL', 0, 0.05);
      lp_gas  = 4.2 + energy_factor + rand('NORMAL', 0, 0.05);
      lincome = 10.8 + 0.004*month + rand('NORMAL', 0, 0.06);

      /* Negatively correlated cross-equation errors (budget tradeoff) */
      shock = rand('NORMAL', 0, 0.010);
      e1 =  shock + rand('NORMAL', 0, 0.006);
      e2 = -shock + rand('NORMAL', 0, 0.006);

      w_elec = a1 + g11*lp_elec + g12*lp_gas + b1*lincome + e1;
      w_gas  = a2 + g21*lp_elec + g22*lp_gas + b2*lincome + e2;

      SAÍDA;
   FIM;

   MANTER month lp_elec lp_gas lincome w_elec w_gas;
   RÓTULO month="Mês"
          lp_elec="Log preço elétrico"
          lp_gas="Log preço do gás"
          lincome="Log renda"
          w_elec="Participação elétrica"
          w_gas="Participação do gás";
EXECUTAR;

PROCEDIMENTO MÉDIAS DADOS=energy n mean std MIN MAX maxdec=3;
   VARIÁVEL w_elec w_gas lp_elec lp_gas lincome;
EXECUTAR;

                                                  The MEANS Procedure

 Variable  Label                           N           Mean     Std Dev     Minimum     Maximum
 ----------------------------------------------------------------------------------------------
 w_elec    Participação elétrica          60          0.440       0.011       0.417       0.464
 w_gas     Participação do gás            60          0.228       0.014       0.198       0.256
 lp_elec   Log preço elétrico             60          4.617       0.142       4.354       4.911
 lp_gas    Log preço do gás               60          4.211       0.162       3.818       4.566
 lincome   Log renda                      60         10.916       0.088      10.723      11.174
 ----------------------------------------------------------------------------------------------




NOTE: DATA energy


NOTE: Wrote energy (60 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Passo 2 — Estimação SUR de base do sistema de demanda

Estimamos ambas as equações de participação conjuntamente. A opção `SUR` diz ao `PROC SYSLIN` para estimar a covariância dos erros entre equações e usá-la para um ajuste eficiente de GLS factível. Duas instruções `MODEL` rotuladas (`elec` e `gas`) definem o sistema; cada uma regride uma participação orçamentária sobre os dois logaritmos de preço e o logaritmo da renda.

O SYSLIN relata as estimativas de parâmetro de cada equação e, ao final, a **Matriz de Covariância Entre Modelos** — a covariância estimada dos erros das duas equações que o SUR explora.

In [2]:
PROCEDIMENTO syslin DADOS=energy ON;
   elec: MODELO w_elec = lp_elec lp_gas lincome;
   gas:  MODELO w_gas  = lp_elec lp_gas lincome;
EXECUTAR;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: Participação elétrica

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: Participação do gás

  Number of Observations                       60
  SSE           


NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: PROC SYSLIN completed.


## Passo 3 — Comparar com MQO equação a equação

Para ver o que o SUR nos traz, reajustamos as mesmas duas equações com mínimos quadrados ordinários (o método padrão, uma equação por vez). O MQO ignora a correlação dos erros entre equações, portanto é consistente, mas menos eficiente que o SUR quando essa correlação é não nula — como é aqui por construção.

Comparar as duas tabelas de coeficientes mostra como as estimativas e seus erros padrão mudam quando a estrutura do sistema é levada em conta.

In [3]:
PROCEDIMENTO syslin DADOS=energy ols;
   elec: MODELO w_elec = lp_elec lp_gas lincome;
   gas:  MODELO w_gas  = lp_elec lp_gas lincome;
EXECUTAR;


                       The SYSLIN Procedure

                  OLS Estimation

  Model elec Dependent Variable: Participação elétrica

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.155544       5.15      0.0000
  LP_ELEC        1      0.112463    0.021989       5.11      0.0000
  LP_GAS         1     -0.097938    0.019300      -5.07      0.0000
  LINCOME        1     -0.042842    0.013702      -3.13      0.0028

  Model gas Dependent Variable: Participação do gás

  Number of Observations                       60
  SSE           


NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (OLS)
NOTE: PROC SYSLIN completed.


## Passo 4 — Impor a teoria econômica e testá-la

A teoria da demanda afirma que os efeitos nominais de preço/renda devem obedecer à **homogeneidade de grau zero**: escalonar todos os preços e a renda deixa a demanda real inalterada, portanto os coeficientes de preço e renda dentro de uma equação somam zero. Impomos isso à participação de eletricidade com uma instrução `RESTRICT`.

O SYSLIN reajusta o sistema sujeito à restrição e relata um teste F de **Restriction Results** sobre se a restrição é consistente com os dados — um teste conjunto direto da hipótese de homogeneidade.

In [4]:
PROCEDIMENTO syslin DADOS=energy ON;
   elec: MODELO w_elec = lp_elec lp_gas lincome;
   gas:  MODELO w_gas  = lp_elec lp_gas lincome;

   /* Homogeneity on the electricity share equation:
      price and income coefficients sum to zero. */
   restrict lp_elec + lp_gas + lincome = 0;
EXECUTAR;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: Participação elétrica

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: Participação do gás

  Number of Observations                       60
  SSE           


NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: PROC SYSLIN completed.


## Passo 5 — Capturar estimativas para o relato de elasticidades

Para um relatório final, persistimos os coeficientes convergidos com `OUTEST=`. O conjunto de dados resultante carrega uma linha por equação com as estimativas de intercepto e inclinação, mais estatísticas de ajuste, que alimentam os cálculos de elasticidade de preço próprio e cruzado da concessionária nas participações médias amostrais. Nós o imprimimos para registro.

In [5]:
PROCEDIMENTO syslin DADOS=energy ON outest=demand_est;
   elec: MODELO w_elec = lp_elec lp_gas lincome;
   gas:  MODELO w_gas  = lp_elec lp_gas lincome;
EXECUTAR;

PROCEDIMENTO IMPRIMIR DADOS=demand_est noobs;
   TÍTULO "Estimativas de coeficientes do sistema de demanda SUR";
EXECUTAR;
TÍTULO;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: Participação elétrica

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: Participação do gás

  Number of Observations                       60
  SSE           


NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: Wrote OUTEST= dataset demand_est (2 rows).
NOTE: PROC SYSLIN completed.
NOTE: PROC PRINT data=demand_est

NOTE: PROC PRINT completed: 2 observations printed, 12 variables


## Interpretando os resultados

**Coerência de sinal.** As estimativas SUR recuperam a estrutura de substituição embutida nos dados. Na equação da participação de eletricidade, o **coeficiente do logaritmo do preço próprio é positivo** (`LP_ELEC` = 0.112), enquanto o **coeficiente do preço cruzado é negativo** (`LP_GAS` = -0.098); na equação da participação de gás o padrão o espelha (próprio `LP_GAS` = 0.114, cruzado `LP_ELEC` = -0.075). Cada efeito de preço próprio e cruzado é estatisticamente significativo (cada `Pr > |t|` abaixo de 0.005), portanto os sinais de substituição são firmemente identificados, e não artefatos de um ajuste ruidoso.

**O SUR explora a correlação entre equações.** A **Matriz de Covariância Entre Modelos** mostra um termo fora da diagonal negativo (-0.000068): as famílias trocam gasto com eletricidade por gasto com gás, exatamente como construído. Como essa covariância é não nula, a estimação conjunta SUR é mais eficiente que ajustar cada equação isoladamente — os erros padrão do SUR no Passo 2 são uniformemente um pouco menores que os erros padrão do MQO equação a equação no Passo 3 (por exemplo, o erro padrão de `LP_ELEC` na equação de gás cai de 0.0264 sob MQO para 0.0255 sob SUR), enquanto as estimativas pontuais permanecem inalteradas. Essa inferência mais precisa é o retorno de modelar o sistema em vez de duas regressões separadas.

**A teoria se sustenta.** Impor **homogeneidade de grau zero** à participação de eletricidade (coeficientes de preço e renda somando zero) via `RESTRICT` produziu um teste F de Restriction Results de 2.14 com `Pr > F` = 0.149. A restrição **não é rejeitada**, portanto a previsão de homogeneidade da teoria da demanda do consumidor é consistente com esta amostra — a concessionária pode levar as estimativas restritas e coerentes com a teoria ao relatório com confiança.

**Uso operacional.** A tabela `OUTEST=` persiste uma linha por equação com as estimativas de intercepto e inclinação e as estatísticas de ajuste. Esses coeficientes convertem-se diretamente em elasticidades de preço próprio e cruzado e em uma elasticidade-renda (de despesa) nas participações médias amostrais (`W_ELEC` ~ 0.44, `W_GAS` ~ 0.23 do Passo 1), dando à concessionária uma base defensável e consistente com a teoria para previsão de demanda e cenários de desenho tarifário.